# GRDP per Capita — Silver
2021–2025 · 


In [0]:
%sql
SET TIME ZONE 'UTC';
DECLARE OR REPLACE VARIABLE grdp_run_id STRING;
DECLARE OR REPLACE VARIABLE grdp_run_ts TIMESTAMP;
DECLARE OR REPLACE VARIABLE grdp_last_id BIGINT;
SET VAR grdp_run_id = CONCAT('silver_grdp_pc_', REPLACE(UUID(), '-', ''));
SET VAR grdp_run_ts = CURRENT_TIMESTAMP();


## 1. Clean and validate


In [0]:
%sql
SELECT ASSERT_TRUE(
    COUNT_IF(
        TRY_CAST(year AS INT) IS NULL
        OR NULLIF(TRIM(_rescued_data), '') IS NOT NULL
    ) = 0,
    'Invalid source years or rescued records; repair Bronze first.'
)
FROM pondolens.dev_bronze.grdp_per_capita;
CREATE OR REPLACE TEMP VIEW grdp_pc_stage AS
WITH normalized AS (
    SELECT
        NULLIF(UPPER(TRIM(region_id)), '') AS source_region_code,
        NULLIF(
            TRIM(REGEXP_REPLACE(region, r'\s+', ' ')),
            ''
        ) AS region_name,
        TRY_CAST(year AS INT) AS year,
        TRY_CAST(
            ROUND(
                TRY_CAST(GRDP_per_capita_amount AS DECIMAL(30, 15)),
                0
            ) AS BIGINT
        ) AS grdp_per_capita,
        LOWER(TRIM(price_basis)) AS source_price_basis,
        NULLIF(
            LOWER(TRIM(geographic_level)),
            ''
        ) AS source_geographic_level
    FROM pondolens.dev_bronze.grdp_per_capita
    WHERE TRY_CAST(year AS INT)
        BETWEEN 2021 AND 2025
)
SELECT
    CASE
        WHEN UPPER(region_name) = 'PHILIPPINES' THEN 'PHL'
        WHEN UPPER(region_name) = 'MIMAROPA REGION' THEN 'IVB'
        ELSE REPLACE(source_region_code, '-', '')
    END AS region_code,
    region_name,
    year,
    grdp_per_capita,
    CASE
        WHEN source_price_basis = 'at current prices' THEN year
        WHEN source_price_basis = 'at constant 2018 prices' THEN 2018
    END AS price_basis,
    CASE
        WHEN UPPER(region_name) = 'PHILIPPINES' THEN 'national'
        ELSE source_geographic_level
    END AS geographic_level
FROM normalized;


In [0]:
%sql
SELECT
    ASSERT_TRUE(COUNT(*) > 0, 'No records in selected scope.') AS nonempty,
    ASSERT_TRUE(
        COUNT_IF(region_code IS NULL OR TRIM(region_code) = ''
        OR region_name IS NULL OR TRIM(region_name) = ''
        OR year IS NULL
        OR grdp_per_capita IS NULL OR grdp_per_capita <= 0
        OR price_basis IS NULL OR (price_basis <> 2018 AND price_basis <> year)
        OR geographic_level IS NULL OR geographic_level NOT IN ('region', 'national')
        OR (geographic_level = 'national'
            AND (region_code <> 'PHL' OR UPPER(region_name) <> 'PHILIPPINES'))
        OR (geographic_level = 'region'
            AND (region_code = 'PHL' OR UPPER(region_name) = 'PHILIPPINES'))) = 0,
        'Invalid source key, amount, price basis or geography.'
    ) AS valid_source,
    ASSERT_TRUE(
        COUNT(*) = COUNT(DISTINCT STRUCT(
            year, region_code, price_basis, geographic_level
        )),
        'Duplicate source business keys.'
    ) AS unique_keys
FROM grdp_pc_stage;


## 2. Prepare target


In [0]:
%sql
CREATE TABLE IF NOT EXISTS pondolens.dev_silver.grdp_per_capita (
    grdp_per_capita_id INT,
    region_code       STRING,
    region_name       STRING,
    year              INT,
    grdp_per_capita   BIGINT,
    price_basis       INT,
    geographic_level STRING,
    processing_run_id STRING,
    processed_at      TIMESTAMP
)
USING DELTA;


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW grdp_pc_target AS
SELECT
    grdp_per_capita_id,
    REPLACE(region_code, '-', '') AS region_code,
    region_name, year, grdp_per_capita, price_basis, geographic_level
FROM pondolens.dev_silver.grdp_per_capita;

SELECT ASSERT_TRUE(
    COUNT(*) = COUNT(DISTINCT grdp_per_capita_id)
    AND COUNT(*) = COUNT(DISTINCT STRUCT(year, region_code, price_basis, geographic_level))
    AND COUNT_IF(
        grdp_per_capita_id <= 100000000 OR grdp_per_capita_id IS NULL
        OR year IS NULL OR year NOT BETWEEN 2021 AND 2025
        OR region_code IS NULL OR TRIM(region_code) = ''
        OR region_name IS NULL OR TRIM(region_name) = ''
        OR grdp_per_capita IS NULL OR grdp_per_capita <= 0
        OR price_basis IS NULL OR (price_basis <> 2018 AND price_basis <> year)
        OR geographic_level IS NULL OR geographic_level NOT IN ('region', 'national')
        OR (geographic_level = 'national' AND
            (region_code <> 'PHL' OR UPPER(region_name) <> 'PHILIPPINES'))
        OR (geographic_level = 'region' AND
            (region_code = 'PHL' OR UPPER(region_name) = 'PHILIPPINES'))
    ) = 0,
    'Invalid existing Silver records or duplicate keys.'
)
FROM grdp_pc_target;

SELECT ASSERT_TRUE(COUNT(*) = 0, 'Existing keys missing from source.')
FROM grdp_pc_target AS t
LEFT ANTI JOIN grdp_pc_stage AS s
    ON t.year = s.year
    AND t.region_code = s.region_code
    AND t.price_basis = s.price_basis
    AND t.geographic_level = s.geographic_level;


## 3. Allocate IDs and merge


In [0]:
%sql
SET VAR grdp_last_id = (
    SELECT COALESCE(
        MAX(CAST(grdp_per_capita_id AS BIGINT)),
        100000000
    )
    FROM pondolens.dev_silver.grdp_per_capita
);
CREATE OR REPLACE TEMP VIEW grdp_pc_new_ids AS
SELECT
    s.year,
    s.region_code,
    s.price_basis,
    s.geographic_level,

    grdp_last_id + ROW_NUMBER() OVER (
        ORDER BY
            s.year,
            s.region_code,
            s.price_basis,
            s.geographic_level
    ) AS new_id

FROM grdp_pc_stage AS s
LEFT ANTI JOIN grdp_pc_target AS t
    ON t.year = s.year
    AND t.region_code = s.region_code
    AND t.price_basis = s.price_basis
    AND t.geographic_level = s.geographic_level;
SELECT ASSERT_TRUE(
    COUNT_IF(
        new_id NOT BETWEEN 100000001 AND 2147483647
    ) = 0,
    'INT ID capacity exceeded; migrate to BIGINT before loading.'
)
FROM grdp_pc_new_ids;
CREATE OR REPLACE TEMP VIEW grdp_pc_merge_source AS
SELECT
    s.*,
    t.grdp_per_capita_id AS existing_id,
    CAST(n.new_id AS INT) AS new_id

FROM grdp_pc_stage AS s
LEFT JOIN grdp_pc_target AS t
    ON t.year = s.year
    AND t.region_code = s.region_code
    AND t.price_basis = s.price_basis
    AND t.geographic_level = s.geographic_level

LEFT JOIN grdp_pc_new_ids AS n
    ON s.year = n.year
    AND s.region_code = n.region_code
    AND s.price_basis = n.price_basis
    AND s.geographic_level = n.geographic_level;
SELECT ASSERT_TRUE(
    COUNT_IF(
        existing_id IS NULL AND new_id IS NULL
    ) = 0,
    'New record has no allocated ID.'
)
FROM grdp_pc_merge_source;


In [0]:
%sql
MERGE INTO pondolens.dev_silver.grdp_per_capita AS t
USING grdp_pc_merge_source AS s
    ON t.year = s.year
    AND REPLACE(t.region_code, '-', '') = s.region_code
    AND t.price_basis = s.price_basis
    AND t.geographic_level = s.geographic_level
WHEN MATCHED AND (
    NOT (t.region_code <=> s.region_code)
    OR NOT (t.region_name <=> s.region_name)
    OR NOT (t.grdp_per_capita <=> s.grdp_per_capita)
)
THEN UPDATE SET
    t.region_code = s.region_code,
    t.region_name             = s.region_name,
    t.grdp_per_capita          = s.grdp_per_capita,
    t.processing_run_id        = grdp_run_id,
    t.processed_at             = grdp_run_ts
WHEN NOT MATCHED THEN INSERT (
    grdp_per_capita_id,
    region_code,
    region_name,
    year,
    grdp_per_capita,
    price_basis,
    geographic_level,
    processing_run_id,
    processed_at
)
VALUES (
    s.new_id,
    s.region_code,
    s.region_name,
    s.year,
    s.grdp_per_capita,
    s.price_basis,
    s.geographic_level,
    grdp_run_id,
    grdp_run_ts
);


## 4. Validate result


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW grdp_pc_differences AS
WITH actual AS (
    SELECT
        region_code,
        region_name,
        year,
        grdp_per_capita,
        price_basis,
        geographic_level
    FROM pondolens.dev_silver.grdp_per_capita
    WHERE year BETWEEN 2021 AND 2025
),
expected AS (
    SELECT
        region_code,
        region_name,
        year,
        grdp_per_capita,
        price_basis,
        geographic_level
    FROM grdp_pc_stage
),
missing AS (
    SELECT * FROM expected
    EXCEPT ALL
    SELECT * FROM actual
),
extra AS (
    SELECT * FROM actual
    EXCEPT ALL
    SELECT * FROM expected
)
SELECT 'MISSING_OR_CHANGED' AS issue, *
FROM missing

UNION ALL

SELECT 'EXTRA_OR_CHANGED' AS issue, *
FROM extra;
SELECT ASSERT_TRUE(
    COUNT(*) = 0,
    'Post-load source/Silver reconciliation failed.'
)
FROM grdp_pc_differences;
SELECT ASSERT_TRUE(
    COUNT(*) = COUNT(DISTINCT grdp_per_capita_id),
    'Post-load duplicate IDs.'
)
FROM pondolens.dev_silver.grdp_per_capita;
SELECT
    year,
    COUNT(*) AS records,
    COUNT_IF(processing_run_id = grdp_run_id) AS written_this_run,
    MIN(grdp_per_capita_id) AS min_id,
    MAX(grdp_per_capita_id) AS max_id
FROM pondolens.dev_silver.grdp_per_capita
GROUP BY year
ORDER BY year;


In [0]:
%sql
SELECT *
FROM pondolens.dev_silver.grdp_per_capita
ORDER BY grdp_per_capita_id;